# Weekly player descriptions

Run this notebook from the repository root, after running the normal data pipeline. It reads the player reference, completed gameweek goal CSVs, and the pipeline's `player_pics.csv`; it generates descriptions only for players who have scored or been picked, using a local Ollama model, and saves one CSV to `data/output/<data_source>/player_descriptions.csv`.

## One-time setup

1. Install Ollama from [ollama.com](https://ollama.com/) and start it.
2. In a terminal, download the model once with `ollama pull llama3.2:3b`.
3. Run this notebook locally. It makes requests only to Ollama on your own computer (`localhost:11434`); there is no hosted LLM API.

The model runs on your computer and may take a few minutes to describe the eligible players. Each successful run overwrites the same CSV, avoiding a growing collection of files. The CSV is saved alongside the pipeline reports and is not loaded by Streamlit.

In [19]:
import json
import re
import urllib.error
import urllib.request
from datetime import datetime
from pathlib import Path

import pandas as pd

from config import data_source

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").is_file():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")

MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"
RECENT_GAMEWEEKS = 3

data_dir = PROJECT_ROOT / "data"
input_dir = data_dir / "input" / data_source
goals_dir = input_dir / "player_goals"
ownership_path = data_dir / "output" / data_source / "player_pics.csv"
reference_path = input_dir / "player_reference.csv"

print(f"Data source: {data_source}")
print(f"Reference: {reference_path}")
print(f"Player picks: {ownership_path}")

Data source: actual
Reference: c:\Users\tomkn\Documents\wsl_game\data\input\actual\player_reference.csv
Player picks: c:\Users\tomkn\Documents\wsl_game\data\output\actual\player_pics.csv


## Load and validate the CSV inputs

The most recent gameweek is found from the available files. Gameweek files must be numbered consecutively from GW 1. A missing player row means zero goals recorded in that gameweek; a blank `goals` cell follows the existing pipeline rule and counts as one goal.

In [20]:
reference = pd.read_csv(reference_path, dtype={"player_id": "string"})
required_reference_columns = {"player_id", "full_name", "team"}
if not required_reference_columns.issubset(reference.columns):
    raise ValueError(f"Reference CSV must contain columns: {sorted(required_reference_columns)}")

reference = reference[["player_id", "full_name", "team"]].copy()
reference["player_id"] = reference["player_id"].str.strip()
if reference.empty or reference[["full_name", "team"]].isna().any().any():
    raise ValueError("The player reference must contain players with names and teams.")
if reference[["full_name", "team"]].apply(lambda column: column.str.strip().eq("")).any().any():
    raise ValueError("The player reference contains an empty player name or team.")
if reference["player_id"].isna().any() or reference["player_id"].eq("").any():
    raise ValueError("Every reference player must have a player_id.")
if reference["player_id"].duplicated().any():
    raise ValueError("The player reference contains duplicate player_id values.")

gameweek_files = {}
for path in goals_dir.glob("GW_*.csv"):
    match = re.fullmatch(r"GW_(\d+)", path.stem)
    if match is None:
        raise ValueError(f"Unexpected gameweek filename: {path.name}")
    gameweek = int(match.group(1))
    if gameweek in gameweek_files:
        raise ValueError(f"More than one file found for GW {gameweek}.")
    gameweek_files[gameweek] = path

if not gameweek_files:
    raise FileNotFoundError(f"No GW_<number>.csv files found in {goals_dir}")

latest_gameweek = max(gameweek_files)
expected_gameweeks = set(range(1, latest_gameweek + 1))
if set(gameweek_files) != expected_gameweeks:
    missing_gameweeks = sorted(expected_gameweeks - set(gameweek_files))
    raise ValueError(f"Gameweek files are not consecutive; missing: {missing_gameweeks}")

player_ids = set(reference["player_id"])
goals_by_gameweek = {}
for gameweek, path in sorted(gameweek_files.items()):
    goals = pd.read_csv(path, dtype={"player_id": "string"})
    if "player_id" not in goals.columns:
        raise ValueError(f"{path.name} must contain a player_id column.")
    if "goals" not in goals.columns:
        goals["goals"] = 1

    goals["player_id"] = goals["player_id"].str.strip()
    if goals["player_id"].isna().any() or goals["player_id"].eq("").any():
        raise ValueError(f"{path.name} contains an empty player_id.")
    if goals["player_id"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate player_id values.")

    goals["goals"] = goals["goals"].replace(r"^\s*$", pd.NA, regex=True).fillna(1)
    numeric_goals = pd.to_numeric(goals["goals"], errors="coerce")
    if numeric_goals.isna().any() or (numeric_goals < 0).any() or (numeric_goals % 1 != 0).any():
        raise ValueError(f"{path.name} contains an invalid goals value.")

    unknown_ids = set(goals["player_id"]) - player_ids
    if unknown_ids:
        raise ValueError(f"{path.name} contains player_id values missing from the reference: {sorted(unknown_ids)}")
    goals_by_gameweek[gameweek] = dict(zip(goals["player_id"], numeric_goals.astype(int)))

ownership = pd.read_csv(ownership_path, dtype={"player_id": "string"})
if not {"player_id", "num_picks"}.issubset(ownership.columns):
    raise ValueError("player_pics.csv must contain player_id and num_picks columns.")
ownership["player_id"] = ownership["player_id"].str.strip()
if ownership["player_id"].isna().any() or ownership["player_id"].eq("").any():
    raise ValueError("player_pics.csv contains an empty player_id.")
if ownership["player_id"].duplicated().any():
    raise ValueError("player_pics.csv contains duplicate player_id values.")
unknown_owner_ids = set(ownership["player_id"]) - player_ids
if unknown_owner_ids:
    raise ValueError(f"player_pics.csv contains IDs missing from the reference: {sorted(unknown_owner_ids)}")

owner_counts = pd.to_numeric(ownership["num_picks"], errors="coerce")
if owner_counts.isna().any() or (owner_counts < 0).any() or (owner_counts % 1 != 0).any():
    raise ValueError("player_pics.csv contains an invalid num_picks value.")
owner_counts = dict(zip(ownership["player_id"], owner_counts.astype(int)))

print(f"Validated {len(reference)} players and GW 1–{latest_gameweek}.")

Validated 348 players and GW 1–6.


## Prepare player statistics

Python calculates the statistics and ranks. The model receives only goal numbers—not player names, team names, ownership, participant data, or access to files/tools. Python appends the verified owner count and any top-three ownership or goalscorer rank after the model writes the goal description. Tied players share a rank; zero picks and zero goals are not ranked.

In [21]:
player_facts = []
for player_id in reference["player_id"]:
    weekly_goals = [goals_by_gameweek[gw].get(player_id, 0) for gw in range(1, latest_gameweek + 1)]
    recent_goals = sum(weekly_goals[-RECENT_GAMEWEEKS:])
    facts = {
        "player_id": player_id,
        "total_goals": sum(weekly_goals),
        "completed_gameweeks": latest_gameweek,
        "recent_gameweeks": min(RECENT_GAMEWEEKS, latest_gameweek),
        "recent_goals": recent_goals,
        "owner_count": owner_counts.get(player_id, 0),
        "first_three_goals": sum(weekly_goals[:3]) if latest_gameweek >= 6 else None,
    }
    player_facts.append(facts)

picked_counts = [facts["owner_count"] for facts in player_facts if facts["owner_count"] > 0]
goal_totals = [facts["total_goals"] for facts in player_facts if facts["total_goals"] > 0]
owner_rank_by_count = {
    count: 1 + sum(other_count > count for other_count in picked_counts)
    for count in set(picked_counts)
}
goals_rank_by_total = {
    total: 1 + sum(other_total > total for other_total in goal_totals)
    for total in set(goal_totals)
}
for facts in player_facts:
    facts["owner_rank"] = owner_rank_by_count.get(facts["owner_count"])
    facts["owner_rank_tie_count"] = picked_counts.count(facts["owner_count"])
    facts["goals_rank"] = goals_rank_by_total.get(facts["total_goals"])
    facts["goals_rank_tie_count"] = goal_totals.count(facts["total_goals"])

player_facts = [facts for facts in player_facts if facts["total_goals"] > 0 or facts["owner_count"] > 0]
print(f"Prepared descriptions for {len(player_facts)} players with goals or picks.")



Prepared descriptions for 84 players with goals or picks.


In [ ]:
def generate_description(facts):
    prompt = (
        "Write one concise, neutral English goal sentence of at most 15 words about a fantasy-game player, "
        "using only the supplied goal facts. If total_goals is zero, say the player has not scored yet. "
        "If first_three_goals is null, state the goal facts without claiming a trend. Otherwise, compare "
        "the first and recent three-gameweek totals only when they support a factual trend. Do not mention "
        "owners, rankings, or infer appearances, injuries, ability, or future performance. Python will append "
        "the verified owner count and any ranking highlights, so keep this sentence short. Return only a JSON object with one string field named "
        f"description. Facts: {json.dumps({key: value for key, value in facts.items() if key not in {'player_id', 'owner_count', 'owner_rank', 'owner_rank_tie_count', 'goals_rank', 'goals_rank_tie_count'}})}"
    )
    request_body = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "format": {
            "type": "object",
            "properties": {"description": {"type": "string"}},
            "required": ["description"],
        },
        "options": {"temperature": 0, "num_predict": 80},
    }
    request = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(request_body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            result = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(
            "Could not reach local Ollama. Start Ollama and confirm llama3.2:3b is installed."
        ) from error

    try:
        generated = json.loads(result["response"])
    except (KeyError, TypeError, json.JSONDecodeError) as error:
        raise ValueError(f"Ollama returned invalid JSON for player_id {facts['player_id']}.") from error
    description = generated.get("description") if isinstance(generated, dict) else None
    if not isinstance(description, str):
        raise ValueError(f"Ollama returned no description string for player_id {facts['player_id']}.")
    description = description.strip()
    if not description:
        raise ValueError(f"Ollama returned an empty description for player_id {facts['player_id']}.")
    description = description.rstrip(".!? ")
    owner_word = "manager" if facts["owner_count"] == 1 else "managers"
    description += f"; selected by {facts['owner_count']} {owner_word}"
    ranking_highlights = []
    owner_rank_labels = {1: "most-owned", 2: "2nd most-owned", 3: "3rd most-owned"}
    if facts["owner_rank"] in owner_rank_labels:
        owner_highlight = owner_rank_labels[facts["owner_rank"]]
        if facts["owner_rank_tie_count"] > 1:
            owner_highlight = f"joint {owner_highlight}"
        ranking_highlights.append(owner_highlight)

    goals_rank_labels = {1: "top scorer", 2: "2nd-highest scorer", 3: "3rd-highest scorer"}
    if facts["goals_rank"] in goals_rank_labels:
        goals_highlight = goals_rank_labels[facts["goals_rank"]]
        if facts["goals_rank_tie_count"] > 1:
            goals_highlight = f"joint {goals_highlight}"
        ranking_highlights.append(goals_highlight)

    if ranking_highlights:
        description += f"; {', '.join(ranking_highlights)}"
    description += "."

    words = description.split()
    if not description or len(words) > 30:
        raise ValueError(
            f"Description for {facts['player_id']} has {len(words)} words; the maximum is 30. "
            "The model's goal sentence must be shorter to leave room for the verified owner count and ranking highlights."
        )
    if description[-1] not in ".!?" or re.search(r"[.!?]", description[:-1]):
        raise ValueError(f"Description for {facts['player_id']} must be one sentence.")
    owner_phrase = f"selected by {facts['owner_count']} {owner_word}"
    if owner_phrase not in description.casefold():
        raise ValueError(f"Description for {facts['player_id']} is missing its verified owner count.")
    return description

descriptions = []
for index, facts in enumerate(player_facts, start=1):
    descriptions.append(generate_description(facts))
    if index % 25 == 0 or index == len(player_facts):
        print(f"Generated {index}/{len(player_facts)} descriptions.")

print(f"Generated {len(descriptions)} descriptions.")

## Review and save

The CSV is written only after every eligible player has a valid description. It overwrites `data/output/<data_source>/player_descriptions.csv`; Streamlit does not load this file.

In [ ]:
eligible_player_ids = {facts["player_id"] for facts in player_facts}
output = reference[reference["player_id"].isin(eligible_player_ids)].copy()
if len(descriptions) != len(output) or set(output["player_id"]) != eligible_player_ids:
    raise ValueError("Generated description count does not match the players with goals or picks.")

output["description"] = descriptions
output["data_source"] = data_source
output["through_gameweek"] = latest_gameweek
output["generated_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
output["model"] = MODEL

if output["player_id"].duplicated().any() or output["description"].isna().any():
    raise ValueError("Output CSV failed completeness or uniqueness validation.")

output_dir = data_dir / "output" / data_source
output_path = output_dir / "player_descriptions.csv"

output.to_csv(output_path, index=False)
print(f"Saved {len(output)} player descriptions to {output_path}")
display(output[["player_id", "full_name", "description"]])

Saved 84 player descriptions to c:\Users\tomkn\Documents\wsl_game\data\local\player_descriptions\player_descriptions_actual_GW6_20261023_150728_597329.csv


,player_id,full_name,description
0,WSL_0018,Alessia Russo,The player has not scored any goals in the fan...
2,WSL_0016,Chloe Kelly,The player has not scored any goals in the fan...
5,WSL_0344,Georgia Stanway,The player has not scored any goals in the fan...
9,WSL_0012,Kim Little,The player has not scored any goals in the fan...
12,WSL_0353,Lisa Baum,The fantasy-game player has scored 1 goal in 6...
...,...,...,...
317,WSL_0289,Olivia Holdt,The fantasy-game player has scored 3 goals in ...
319,WSL_0307,Shekiera Martinez,The fantasy-game player has scored 1 goal in 6...
320,WSL_0287,Signe Gaupset,The fantasy-game player has scored 2 goals in ...
322,WSL_0308,Victoria Pelova,The fantasy-game player has scored 5 total goa...


In [ ]:
output_path

WindowsPath('c:/Users/tomkn/Documents/wsl_game/data/local/player_descriptions/player_descriptions_actual_GW6_20261023_150728_597329.csv')